# Report

In [ ]:
import os
import sys

import pandas as pd

rootpath = os.path.dirname(os.getcwd())
sys.path.append(os.path.join(rootpath, "scripts"))
sys.path.append(os.path.join(rootpath, "src"))

In [ ]:
from gcp_functions import load_json_from_gcs, load_pickle_from_gcs, read_file_as_df
from load_configs import Configs

from data import (
    build_features,
    create_X_y_multistep,
)

configs = Configs()

## Helper Functions

In [ ]:
def prepare_data(data_path: dict, params: dict, estimator: object) -> pd.DataFrame:
    df = read_file_as_df(
        data_path["project"],
        data_path["bucket"],
        data_path["prefix"] + "/" + data_path["fname"],
    )

    TARGET = "returns"
    df_feats, _features2scale = build_features(
        df, lags=int(params["lags"]), split="train", CldrFeats=params["CldrFeats"]
    )
    X, y = create_X_y_multistep(df_feats, steps=int(params["steps"]), target=TARGET)
    y_hat = estimator.predict(X)
    data = X.copy()
    data["target"] = y["y_step_1"]
    data["prediction"] = y_hat[:, 0]
    return data

# Create Report

In [ ]:
from evidently import DataDefinition, Dataset, Report
from evidently.metrics import DriftedColumnsCount, MissingValueCount, ValueDrift

In [ ]:
# Read ref_data, ref_params and ref_estimator from GCS
ref_data = read_file_as_df(
    configs.cloud["gcs"]["project"],
    configs.cloud["gcs"]["data_monitoring_bucket"],
    "ref_data_model/data.parquet",
)

ref_params = load_json_from_gcs(
    configs.cloud["gcs"]["project"],
    configs.cloud["gcs"]["data_monitoring_bucket"],
    "ref_data_model/params.json",
)

ref_estimator = load_pickle_from_gcs(
    configs.cloud["gcs"]["project"],
    configs.cloud["gcs"]["data_monitoring_bucket"],
    "ref_data_model/model.pkl",
)

In [ ]:
# Prepare new data, e.g., based on some new data in gcs dev bucket
data_new_path = {
    "project": configs.cloud["gcs"]["project"],
    "bucket": configs.cloud["gcs"]["data_monitoring_bucket"],
    "prefix": "cleaned_samples_dev",
    "fname": "Kaggle_Access_2025-07-28_WSPall_from_2020-07-28.parquet",
}

# Let's use (we don't have to) the ref params and estimator to prepare new data
new_data = prepare_data(data_new_path, ref_params, ref_estimator)

In [ ]:
# convert the multi-index to single-index df
ref_data_flat = ref_data.reset_index(level="Ticker")
new_data_flat = new_data.reset_index(level="Ticker")

In [ ]:
num_features = ref_data.columns.to_list()
data_definition = DataDefinition(
    numerical_columns=num_features, categorical_columns=["Ticker"]
)

ref_dataset = Dataset.from_pandas(ref_data_flat, data_definition)

new_dataset = Dataset.from_pandas(new_data_flat, data_definition)

In [ ]:
report = Report(
    metrics=[
        ValueDrift(column="prediction"),
        DriftedColumnsCount(),
        MissingValueCount(column="prediction"),
    ]
)

In [ ]:
snapshot = report.run(reference_data=ref_dataset, current_data=new_dataset)

In [ ]:
snapshot

In [ ]:
result = snapshot.dict()
result